In [ ]:
"""
Fully Modern LangChain RAG Example (No langchain.chains Dependency)
==================================================================

This version avoids:
    from langchain.chains import RetrievalQAWithSourcesChain

because in some newer LangChain installations, the `langchain.chains`
module may not be included or may be restructured.

Instead, this script uses the recommended LCEL pattern:
- create_stuff_documents_chain
- create_retrieval_chain

This is the modern and preferred approach in current LangChain versions.
"""

import os
from dotenv import load_dotenv

# Modern debugging API
from langchain.globals import set_debug

# OpenAI integrations
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Community integrations
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_community.vectorstores import FAISS

# Text splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Prompts
from langchain_core.prompts import ChatPromptTemplate

# Modern chain constructors
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain


# ==========================================================
# 1. Load API Key
# ==========================================================
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not found.\n"
        "Create a .env file containing:\n"
        "OPENAI_API_KEY=your_openai_api_key_here"
    )


# ==========================================================
# 2. Enable Debugging (Optional)
# ==========================================================
set_debug(False)


# ==========================================================
# 3. Initialize Chat Model
# ==========================================================
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.3,
    max_tokens=500,
)


# ==========================================================
# 4. Load Web Documents
# ==========================================================
urls = [
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html",
]

print("Loading documents...")
loader = UnstructuredURLLoader(urls=urls)
documents = loader.load()

print(f"Loaded {len(documents)} documents.")


# ==========================================================
# 5. Split Documents
# ==========================================================
print("Splitting documents into chunks...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

docs = text_splitter.split_documents(documents)

print(f"Created {len(docs)} chunks.")


# ==========================================================
# 6. Create Embeddings
# ==========================================================
print("Creating embeddings...")

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# ==========================================================
# 7. Build FAISS Vector Store
# ==========================================================
print("Building vector store...")

vector_store = FAISS.from_documents(
    documents=docs,
    embedding=embeddings,
)

# Save locally
vector_store.save_local("faiss_index")

# Reload (optional)
vector_store = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True,
)

print("Vector store ready.")


# ==========================================================
# 8. Create Retriever
# ==========================================================
retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)


# ==========================================================
# 9. Create Prompt
# ==========================================================
prompt = ChatPromptTemplate.from_template(
    """
Answer the user's question using only the provided context.

If the answer is not contained in the context, say you do not know.

At the end, include a section:
SOURCES: <comma-separated list of source URLs>

Context:
{context}

Question:
{input}

Answer:
"""
)


# ==========================================================
# 10. Create Document Chain
# ==========================================================
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt,
)


# ==========================================================
# 11. Create Retrieval Chain
# ==========================================================
retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain,
)


# ==========================================================
# 12. Ask a Question
# ==========================================================
query = "What is the price of Tiago iCNG?"

print(f"\nQuestion: {query}\n")

result = retrieval_chain.invoke({
    "input": query
})


# ==========================================================
# 13. Show Result
# ==========================================================
print("Answer:")
print(result["answer"])


# ==========================================================
# 14. Interactive Loop
# ==========================================================
while True:
    user_query = input("\nAsk a question (or type 'exit'): ").strip()

    if user_query.lower() in {"exit", "quit"}:
        break

    response = retrieval_chain.invoke({
        "input": user_query
    })

    print("\nAnswer:")
    print(response["answer"])

ModuleNotFoundError: No module named 'langchain.globals'